### Samplig + processing patients

In [1]:
import pandas as pd

prediction_results = pd.read_csv('prediction_results_by_hadm.csv')
total_data_full = pd.read_parquet('ind_hosp.parquet')
correct_predictions = prediction_results[prediction_results['correct_prediction'] == True]
test_pairs = correct_predictions[['subject_id', 'hadm_id']].drop_duplicates()
test_pairs_tuple = list(zip(test_pairs['subject_id'], test_pairs['hadm_id']))

total_data = total_data_full[
    total_data_full[['subject_id', 'hadm_id']].apply(tuple, axis=1).isin(test_pairs_tuple)
].copy()
total_data = total_data.merge(
    prediction_results[['subject_id', 'hadm_id', 'predicted_proba']], 
    on=['subject_id', 'hadm_id'], 
    how='left'
)

print(f"Data filtered to test hospitalizations only:")
print(f"  Total rows: {len(total_data)}")
print(f"  Unique patients: {total_data['subject_id'].nunique()}")
print(f"  Unique hospitalizations: {total_data['hadm_id'].nunique()}")

Data filtered to test hospitalizations only:
  Total rows: 237684
  Unique patients: 33311
  Unique hospitalizations: 58385


In [2]:
import numpy as np
threshold = 0.2158
total_data["risk_scaled"] = np.where(
    total_data["predicted_proba"] <= threshold,
    0.5 * total_data["predicted_proba"] / threshold,
    0.5 + 0.5 * (
        (total_data["predicted_proba"] - threshold)
        / (total_data["predicted_proba"].max() - threshold)
    )
)
total_data.head(3)

,subject_id,hadm_id,dischtime,los,age,day,current_date,num_diagnoses,num_chronic,ccsr_FAC021,...,admission_location_INTERNAL TRANSFER TO OR FROM PSYCH,admission_location_PACU,admission_location_PHYSICIAN REFERRAL,admission_location_PROCEDURE SITE,admission_location_TRANSFER FROM HOSPITAL,admission_location_TRANSFER FROM SKILLED NURSING FACILITY,admission_location_WALK-IN/SELF REFERRAL,gender_male,predicted_proba,risk_scaled
0,10000764,27897940,2132-10-19,4,86,1,2132-10-14,76,32,1,...,False,False,False,False,True,False,False,1,0.112961,0.261726
1,10000764,27897940,2132-10-19,4,86,2,2132-10-15,76,32,1,...,False,False,False,False,True,False,False,1,0.112961,0.261726
2,10000764,27897940,2132-10-19,4,86,3,2132-10-16,76,32,1,...,False,False,False,False,True,False,False,1,0.112961,0.261726


In [3]:
# agg_dict = {}

# for col in total_data.columns:
#     if col in ['subject_id', 'hadm_id']:
#         continue
    
#     if col.startswith('icd_'):
#         agg_dict[col] = 'max'
    
#     elif col.startswith('ccsr_'):
#         agg_dict[col] = 'max'
    
#     elif col.startswith('lab_') and col.endswith('_daily'):
#         agg_dict[col] = 'mean'
    
#     elif col in ['cumulative_procedures', 'cumulative_medications']:
#         agg_dict[col] = 'mean'
    
#     elif col in ['num_medications_daily', 'num_procedures_daily']:
#         agg_dict[col] = 'mean'
    
#     elif col in ['gender_male', 'age', 'los', 'num_diagnoses', 'num_chronic', 
#                  'comorbidity_score', 'prior_admissions_12mo', 
#                  'admission_type', 'discharge_location', 'insurance', 'race',
#                  'admission_location']:
#         agg_dict[col] = 'first'
    
#     elif col == 'target_readmission_30d':
#         agg_dict[col] = 'max'
    
#     else:
#         agg_dict[col] = 'first'

#print(f"Aggregating {len(agg_dict)} columns...")
#total_data = total_data.groupby(['subject_id', 'hadm_id']).agg(agg_dict).reset_index()

total_data = (
    total_data
    .sort_values(["subject_id", "hadm_id", "day"])
    .groupby(["subject_id", "hadm_id"])
    .last()
    .reset_index()
)
total_data.head(5)

,subject_id,hadm_id,dischtime,los,age,day,current_date,num_diagnoses,num_chronic,ccsr_FAC021,...,admission_location_INTERNAL TRANSFER TO OR FROM PSYCH,admission_location_PACU,admission_location_PHYSICIAN REFERRAL,admission_location_PROCEDURE SITE,admission_location_TRANSFER FROM HOSPITAL,admission_location_TRANSFER FROM SKILLED NURSING FACILITY,admission_location_WALK-IN/SELF REFERRAL,gender_male,predicted_proba,risk_scaled
0,10000764,27897940,2132-10-19,4,86,4,2132-10-17,76,32,1,...,False,False,False,False,True,False,False,1,0.112961,0.261726
1,10001492,27463908,2136-09-25,1,71,1,2136-09-23,4,4,0,...,False,False,False,False,False,False,False,0,0.064727,0.149970
2,10001667,22672901,2173-08-24,1,86,1,2173-08-22,13,6,1,...,False,False,False,False,True,False,False,0,0.083145,0.192644
3,10001725,25563031,2110-04-14,2,46,2,2110-04-12,36,18,1,...,False,True,False,False,False,False,False,0,0.122675,0.284234
4,10001860,21441082,2188-03-30,3,84,3,2188-03-29,36,9,1,...,False,False,False,False,False,False,False,1,0.081494,0.188818


In [4]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_feature_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    elif feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    elif feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

In [5]:
# def get_patient_analysis(subject_id, hadm_id):
#     patient_delta = total_deltas[
#         (total_deltas['subject_id'] == subject_id) & 
#         (total_deltas['hadm_id'] == hadm_id)
#     ].copy()
#     print(f"Actual risk: {patient_delta['risk_actual_mean'].iloc[0]:.4f}")
#     patient_shap = shap_matrix[shap_matrix['hadm_id'] == hadm_id].copy()
#     shap_features = patient_shap.drop('hadm_id', axis=1).T
#     shap_features.columns = ['shap_value']
#     shap_features['feature'] = shap_features.index
    
#     combined = patient_delta.merge(shap_features, on='feature', how='inner')
#     combined['delta_abs'] = combined['delta_mean'].abs()
#     combined['shap_abs'] = combined['shap_value'].abs()
    
#     return {
#         'patient_id': subject_id,
#         'hadm_id': hadm_id,
#         'n_days': patient_delta['los'].iloc[0],
#         'risk_actual': patient_delta['risk_actual_mean'].iloc[0],
#         'combined': combined,
#         'by_delta': combined.sort_values('delta_abs', ascending=False),
#         'by_shap': combined.sort_values('shap_abs', ascending=False)
#     }

# def interpret_delta(delta, mode_direction):
#     if mode_direction == 'add':
#         if delta > 0:
#             return "without → ↑ risk"
#         elif delta < 0:
#             return "without → ↓ risk"

#     elif mode_direction == 'remove':
#         if delta < 0:
#             return "with → ↓ risk"
#         elif delta > 0:
#             return "with → ↑ risk"

#     elif mode_direction == 'increase_to_mean':
#         if delta < 0:
#             return "lower than mean → ↓ risk"
#         elif delta > 0:
#             return "lower than mean → ↑ risk"
    
#     elif mode_direction == 'reduce_to_mean':
#         if delta < 0:
#             return "higher than mean → ↓ risk"
#         elif delta > 0:
#             return "higher than mean → ↑ risk"
    
#     elif mode_direction == 'low':
#         if delta > 0:
#             return "low → ↑ risk"
#         elif delta < 0:
#             return "low → ↓ risk"

#     elif mode_direction == 'high':
#         if delta > 0:
#             return "high → ↑ risk"
#         elif delta < 0:
#             return "high → ↓ risk"
    
#     elif mode_direction == 'normal':
#         if delta > 0:
#             return "normal range → ↑ risk"
#         elif delta < 0:
#             return "normal range → ↓ risk"

#     elif mode_direction == 'older':
#         if delta < 0:
#             return "older → ↓ risk"
#         elif delta > 0:
#             return "older → ↑ risk"
    
#     elif mode_direction == 'younger':
#         if delta > 0:
#             return "younger → ↑ risk"
#         elif delta < 0:
#             return "younger → ↓ risk"

#     elif mode_direction == 'opposite':
#         if delta > 0:
#             return "↑ risk"
#         elif delta < 0:
#             return "↓ risk"

In [6]:
patient_info = total_data.copy()
icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]

patient_info['icd_count'] = patient_info[icd_cols].sum(axis=1)
patient_info['ccsr_count'] = patient_info[ccsr_cols].sum(axis=1)
patient_info['lab_count'] = patient_info[lab_cols].notna().sum(axis=1)

patient_info = patient_info[
    (patient_info['icd_count'] >= 1) & (patient_info['ccsr_count'] >= 1) &
    (patient_info['lab_count'] >= 1)
].copy()

age_groups = {
    'young': (18, 40),
    'middle': (40, 65),
    'elderly': (65, 80),
    'very_elderly': (80, 120)
}

def get_age_group(age):
    for group, (low, high) in age_groups.items():
        if low <= age < high:
            return group
    return 'unknown'

patient_info = patient_info[['subject_id', 'hadm_id', 'gender_male', 'age', 'los', 'icd_count', 'ccsr_count', 'lab_count', 'target_readmission_30d', 'risk_scaled']]
patient_info['age_group'] = patient_info['age'].apply(get_age_group)

print(f"Patient info created:")
print(f"  Total hospitalizations: {len(patient_info)}")
print(f"  With readmission: {patient_info['target_readmission_30d'].sum():.0f}")
print(f"  Without readmission: {len(patient_info) - patient_info['target_readmission_30d'].sum():.0f}")

def get_stratified_sample(patient_info, sample_size=20, random_state=42):
    readmission_yes = patient_info[patient_info['target_readmission_30d'] == 1]
    readmission_no = patient_info[patient_info['target_readmission_30d'] == 0]
    
    n_total = sample_size
    n_yes = n_total // 2
    n_no = n_total - n_yes
    samples = []

    for group, label, n in [(readmission_yes, 'with_readmission', n_yes), 
                            (readmission_no, 'without_readmission', n_no)]:
       
        age_groups_list = group['age_group'].unique()
        n_per_age = n // len(age_groups_list) if len(age_groups_list) > 0 else 0
        remainder = n - n_per_age * len(age_groups_list)
        
        for i, age_group in enumerate(age_groups_list):
            age_group_patients = group[group['age_group'] == age_group]

            n_age = n_per_age + (1 if i < remainder else 0)
            if n_age == 0:
                continue

            male = age_group_patients[age_group_patients['gender_male'] == 1]
            female = age_group_patients[age_group_patients['gender_male'] == 0]
            
            n_male_age = min(n_age // 2, len(male))
            n_female_age = n_age - n_male_age
            
            if len(male) < n_male_age:
                n_male_age = len(male)
                n_female_age = n_age - n_male_age
            
            if len(female) < n_female_age:
                n_female_age = len(female)
                n_male_age = n_age - n_female_age
            
            if len(male) > 0 and n_male_age > 0:
                sample_male = male.sample(n=min(n_male_age, len(male)), random_state=random_state)
                samples.append(sample_male)
            
            if len(female) > 0 and n_female_age > 0:
                sample_female = female.sample(n=min(n_female_age, len(female)), random_state=random_state)
                samples.append(sample_female)

    if len(samples) > 0:
        sample = pd.concat(samples).sample(frac=1, random_state=random_state)
    else:
        print("No samples selected")
        return pd.DataFrame()
    
    print(f"\nSelected {len(sample)} hospitalizations")
    print(f"   With readmission: {sample['target_readmission_30d'].sum():.0f}")
    print(f"   Without readmission: {len(sample) - sample['target_readmission_30d'].sum():.0f}")
    print(f"   Male: {sample['gender_male'].sum():.0f}")
    print(f"   Female: {len(sample) - sample['gender_male'].sum():.0f}")
    
    print(f"\nAge groups in sample:")
    print(sample['age_group'].value_counts().sort_index())
    
    return sample

SAMPLE_SIZE = 32
sample_patients_df = get_stratified_sample(patient_info, sample_size=SAMPLE_SIZE, random_state=13)
sample_patients_df

Patient info created:
  Total hospitalizations: 34892
  With readmission: 6498
  Without readmission: 28394

Selected 32 hospitalizations
   With readmission: 16
   Without readmission: 16
   Male: 16
   Female: 16

Age groups in sample:
age_group
elderly         8
middle          8
very_elderly    8
young           8
Name: count, dtype: int64


,subject_id,hadm_id,gender_male,age,los,icd_count,ccsr_count,lab_count,target_readmission_30d,risk_scaled,age_group
37151,16380214,20860234,0,50,3,1.0,2,20,0,0.208987,middle
9956,11719852,29694624,0,63,9,2.0,4,20,1,0.634654,middle
106,10017882,20281726,1,86,7,1.0,3,20,1,0.654430,very_elderly
46878,18021792,23272302,1,70,2,2.0,2,20,0,0.164324,elderly
26086,14486442,24893935,0,39,7,1.0,2,20,0,0.310673,young
48764,18360819,27033030,1,71,4,2.0,2,20,1,0.666372,elderly
28460,14898288,21598790,1,39,2,1.0,1,20,0,0.246203,young
16342,12789235,26659628,1,81,6,1.0,1,20,0,0.213961,very_elderly
3715,10652693,28272409,0,75,5,3.0,8,20,0,0.260988,elderly
35630,16124481,28174564,0,30,7,2.0,4,20,1,0.829577,young


In [7]:
def get_patient_full_data(subject_id, hadm_id, total_data, risk_score=False):
    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) &
        (total_data['hadm_id'] == hadm_id)
    ].sort_values("day")

    patient = patient_raw.iloc[-1].to_dict()

    json_context = {
        "demographics": {},
        "diagnoses": {},
        "laboratory_values": {},
        "clinical_indicators": {}
    }
    if risk_score:
        json_context = {
            "risk_score": {},
            #"readmission": {},
            "demographics": {},
            "diagnoses": {},
            "laboratory_values": {},
            "clinical_indicators": {}
        }
        
        json_context["risk_score"] = round(patient.get("risk_scaled", None), 2)
        #json_context["readmission"] = "True" if patient.get("true_class", None) == 1 else "False"

    json_context["demographics"]["gender"] = (
        "Male" if patient["gender_male"] == 1 else "Female"
    )
    json_context["demographics"]["age"] = (patient["age"])
    json_context["demographics"]["length_of_stay"] = (patient["los"])

    race_cols = [c for c in total_data.columns if c.startswith("race_")]

    for col in race_cols:
        if patient.get(col) == 1:
            json_context["demographics"]["race"] = (
                col.replace("race_", "")
                .replace("_", " ")
                .title()
            )
            break

    insurance_cols = [col for col in total_data.columns if col.startswith('insurance_')]
    for col in insurance_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["insurance"] = col.replace('insurance_', '')
            break
    if 'insurance' not in json_context["demographics"]:
        json_context["demographics"]["insurance"] = 'Unknown'

    admission_type_cols = [col for col in total_data.columns if col.startswith('admission_type_')]
    for col in admission_type_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_type"] = col.replace('admission_type_', '').replace('_', ' ').title()
            break
    if 'admission_type' not in json_context["demographics"]:
        json_context["demographics"]["admission_type"] = 'Unknown'

    discharge_cols = [col for col in total_data.columns if col.startswith('discharge_location_')]
    for col in discharge_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["discharge_location"] = col.replace('discharge_location_', '').replace('_', ' ').title()
            break
    if 'discharge_location' not in json_context["demographics"]:
        json_context["demographics"]["discharge_location"] = 'Unknown'

    adm_location_cols = [col for col in total_data.columns if col.startswith('admission_location_')]
    for col in adm_location_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_location"] = col.replace('admission_location_', '').replace('_', ' ').title()
            break
    if 'admission_location' not in json_context["demographics"]:
        json_context["demographics"]["admission_location"] = 'Unknown'

    icd_list = []
    icd_cols = [c for c in total_data.columns if c.startswith("icd_")]
    for col in icd_cols:
        if patient.get(col) == 1:
            icd_list.append(map_feature_name(col))
    json_context["diagnoses"]["icd"] = icd_list

    ccsr_list = []
    ccsr_cols = [c for c in total_data.columns if c.startswith("ccsr_")]
    for col in ccsr_cols:
        if patient.get(col) == 1:
            ccsr_list.append(map_feature_name(col))
    json_context["diagnoses"]["ccsr"] = ccsr_list

    lab_cols = [
        c for c in total_data.columns
        if c.startswith("lab_") and c.endswith("_daily")]
    for col in lab_cols:
        value = patient.get(col)
        if pd.notna(value):
            json_context["laboratory_values"][map_feature_name(col)] = round(float(value), 2)

    other_features = [
        "num_diagnoses",
        "num_chronic",
        "comorbidity_score",
        "num_medications_daily",
        "prior_admissions_12mo",
        "cumulative_procedures",
        "cumulative_medications",
        "num_procedures_daily"
    ]

    for feat in other_features:
        value = patient.get(feat)
        if pd.notna(value):
            json_context["clinical_indicators"][feat] = float(value)

    return {
        "subject_id": int(subject_id),
        "hadm_id": int(hadm_id),
        "json_context": json_context
    }

In [8]:
import json
import numpy as np
import pandas as pd

def convert_to_serializable(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, pd.Series):
        return obj.to_dict()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, pd.DataFrame):
        return obj.to_dict('records')
    elif pd.isna(obj):
        return None
    else:
        return obj

In [9]:
import random
import copy

def convert_to_unstructured_text(patient_record):
    demographics = patient_record['demographics']
    diagnoses = patient_record['diagnoses']
    labs = patient_record['laboratory_values']
    clinical = patient_record['clinical_indicators']
    
    gender = demographics['gender'].lower()
    pronoun = 'He' if gender == 'male' else 'She'
    
    text = f"The patient is a {demographics['age']}-year-old {gender}. "
    text += f"{pronoun} is of {demographics['race']} race and has {demographics['insurance']} insurance. "
    text += f"The length of hospital stay is {demographics['length_of_stay']} days. "
    
    text += f"{pronoun} was admitted through {demographics['admission_location']} with admission type {demographics['admission_type']}. "
    text += f"The discharge location is {demographics['discharge_location']}. "
    
    if diagnoses['icd']:
        text += f"The patient has the following ICD diagnoses: "
        text += ", ".join(diagnoses['icd'])
        text += ". "
    else:
        text += "No ICD diagnoses are recorded. "
    
    if diagnoses['ccsr']:
        text += f"The CCSR categories include: "
        text += ", ".join(diagnoses['ccsr'])
        text += ". "
    else:
        text += "No CCSR categories are recorded. "
    
    if labs:
        text += f"Laboratory values show: "
        lab_items = []
        for lab, value in labs.items():
            lab_items.append(f"{lab} of {value:.2f}")
        text += ", ".join(lab_items)
        text += ". "
    else:
        text += "No laboratory values are available. "
    
    if any(v is not None for v in clinical.values()):
        text += f"Clinical indicators: "
        clinical_items = []
        for key, value in clinical.items():
            if value is not None:
                clinical_items.append(f"{key} is {value:.2f}")
        text += ", ".join(clinical_items)
        text += ". "
    
    return text

def convert_to_empty_card(patient_record):
    demographics = patient_record['demographics']
    
    empty_card = {
        'demographics': {
            'age': demographics.get('age'),
            'gender': demographics.get('gender'),
            'race': demographics.get('race', 'Unknown'),
            'insurance': demographics.get('insurance', 'Unknown'),
            'admission_type': demographics.get('admission_type', 'Unknown'),
            'admission_location': demographics.get('admission_location', 'Unknown'),
        }
    }
    
    return empty_card

def convert_to_incomplete_card(patient_clean, missing_fraction=0.3, random_state=42):
    random.seed(random_state)
    incomplete = copy.deepcopy(patient_clean)
    incomplete.pop('risk_score')
    removable = []

    for lab in incomplete["laboratory_values"]:
        removable.append(("laboratory_values", lab))
    
    for feat in incomplete["clinical_indicators"]:
        removable.append(("clinical_indicators", feat))

    for dx in incomplete["diagnoses"]["icd"]:
        removable.append(("icd", dx))
    
    for dx in incomplete["diagnoses"]["ccsr"]:
        removable.append(("ccsr", dx))
    
    n_remove = max(1, int(len(removable) * missing_fraction))
    remove = random.sample(removable, n_remove)

    for category, value in remove:
        if category == "laboratory_values":
            incomplete["laboratory_values"][value] = None

        elif category == "clinical_indicators":
            incomplete["clinical_indicators"][value] = None

        elif category == "icd":
            incomplete["diagnoses"]["icd"].remove(value)

        elif category == "ccsr":
            incomplete["diagnoses"]["ccsr"].remove(value)
    return incomplete
   
def convert_to_row_column_format(patient_record):
    rows = []
    rows.extend([
        {
            "feature_category": "demographics",
            "feature_name": feature,
            "value": value
        }
        for feature, value in patient_record["demographics"].items()
    ])

    rows.extend([
        {
            "feature_category": "diagnosis",
            "feature_name": f"ICD",
            "value": diagnosis
        }
        for i, diagnosis in enumerate(patient_record["diagnoses"]["icd"], start=1)
    ])

    rows.extend([
        {
            "feature_category": "diagnosis",
            "feature_name": f"CCSR",
            "value": diagnosis
        }
        for i, diagnosis in enumerate(patient_record["diagnoses"]["ccsr"], start=1)
    ])

    rows.extend([
        {
            "feature_category": "laboratory",
            "feature_name": feature,
            "value": value
        }
        for feature, value in patient_record["laboratory_values"].items()
    ])

    rows.extend([
        {
            "feature_category": "clinical indicator",
            "feature_name": feature,
            "value": value
        }
        for feature, value in patient_record["clinical_indicators"].items()
    ])

    return pd.DataFrame(rows)

def convert_to_long_list(patient_clean, subject_id, hadm_id, random_state=42):
    random.seed(random_state)
    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) & 
        (total_data['hadm_id'] == hadm_id)
    ]
    
    if len(patient_raw) == 0:
        return None
    
    patient = patient_raw.iloc[0].to_dict()
    
    all_features = {
        'all_features': []
    }
    demographics = patient_clean['demographics']
    demographic_features = [
        ('age', demographics.get('age')),
        ('gender', 'Male' if demographics.get('gender_male', 0) == 1 else 'Female'),
        ('race', demographics.get('race', 'Unknown')),
        ('insurance', demographics.get('insurance', 'Unknown')),
        ('length_of_stay', demographics.get('length_of_stay', 0)),
        ('admission_type', demographics.get('admission_type', 'Unknown')),
        ('admission_location', demographics.get('admission_location', 'Unknown')),
        ('discharge_location', demographics.get('discharge_location', 'Unknown'))
    ]
    
    for name, value in demographic_features:
        all_features['all_features'].append({
            'feature_category': 'demographics',
            'feature_name': name,
            'value': value
        })
    
    icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
    for col in icd_cols:
        value = patient.get(col, 0)
        display_name = map_feature_name(col)
        all_features['all_features'].append({
            'feature_category': 'icd_diagnosis',
            'feature_name': display_name,
            'value': int(value)
        })
    
    ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
    for col in ccsr_cols:
        value = patient.get(col, 0)
        display_name = map_feature_name(col)
        all_features['all_features'].append({
            'feature_category': 'ccsr_category',
            'feature_name': display_name,
            'value': value
        })
    
    lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]
    for col in lab_cols:
        value = patient.get(col)
        display_name = map_feature_name(col)
        all_features['all_features'].append({
            'feature_category': 'laboratory',
            'feature_name': display_name,
            'value': value
        })
    
    clinical_features = [
        'num_diagnoses', 'num_chronic', 'comorbidity_score',
        'num_medications_daily', 'prior_admissions_12mo',
        'cumulative_procedures', 'cumulative_medications',
        'num_procedures_daily'
    ]
    
    for feat in clinical_features:
        value = patient.get(feat)
        all_features['all_features'].append({
            'feature_category': 'clinical_indicator',
            'feature_name': feat,
            'value': value
        })
    
    return convert_to_serializable(all_features)

In [10]:
def save_all_formats_to_single_json(sample_patients_df, total_data, output_file='all_patients_formats.json'):
    
    all_patients_formatted = []
    failed_patients = []
    
    print(f"Processing {len(sample_patients_df)} patients...")
    
    for idx, row in sample_patients_df.iterrows():
        subject_id = row['subject_id']
        hadm_id = row['hadm_id']
        
        try:
            clean_data = get_patient_full_data(subject_id, hadm_id, total_data, True)
            clean_data = convert_to_serializable(clean_data)
            clean_data = clean_data['json_context']
            unstructured_context = convert_to_unstructured_text(clean_data)
            empty_context = convert_to_empty_card(clean_data)
            row_column_context = convert_to_row_column_format(clean_data)
            incomplete_context = convert_to_incomplete_card(clean_data, random_state=42)
            long_list_context = convert_to_long_list( 
                clean_data, 
                subject_id, 
                hadm_id, 
                random_state=42
            )

            risk_score = clean_data['risk_score']
            clean_data.pop('risk_score')
            patient_entry = {
                'subject_id': subject_id,
                'hadm_id': hadm_id,
                'risk_score': risk_score,
                'json_context': clean_data,
                'unstructured_context': unstructured_context,
                'empty_context': empty_context,
                'row_column_context': row_column_context,
                'long_list_context': long_list_context['all_features'],
                'incomplete_context': incomplete_context
            }

            # if analysis:
            #     shap_delta = {
            #         'risk_actual': float(analysis.get('risk_actual_mean', 0)),
            #         'n_days': int(analysis.get('los', 0)),
            #         'readmission': clean_data.get('readmission'),
            #         'shap_values': {},
            #         'delta_values': {},
            #         'delta_interpretations': {}
            #     }
                
            #     if 'by_shap' in analysis and len(analysis['by_shap']) > 0:
            #         for _, row_df in analysis['by_shap'].iterrows():
            #             feature = row_df.get('feature_display', row_df.get('feature'))
            #             shap_delta['shap_values'][feature] = float(row_df.get('shap_value', 0))
                
            #     if 'by_delta' in analysis and len(analysis['by_delta']) > 0:
            #         for _, row_df in analysis['by_delta'].iterrows():
            #             feature = row_df.get('feature_display', row_df.get('feature'))
            #             shap_delta['delta_values'][feature] = float(row_df.get('delta_mean', 0))
            #             if 'interpretation' in row_df:
            #                 shap_delta['delta_interpretations'][feature] = row_df['interpretation']
                
            #     patient_entry['analysis'] = shap_delta
            
            all_patients_formatted.append(patient_entry)
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id}")
            
        except Exception as e:
            failed_patients.append(f"{subject_id}_{hadm_id}")
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id} - ERROR: {e}")
    
    output_data = {
        'total_patients': len(all_patients_formatted),
        'failed_patients': failed_patients,
        'patients': all_patients_formatted
    }
    
    output_data = convert_to_serializable(output_data)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    
    print(f"\nSuccess: {len(all_patients_formatted)} patients saved to: {output_file}")
    if failed_patients:
        print(f"Failed: {len(failed_patients)} patients")
    
    return output_data

all_formats_data = save_all_formats_to_single_json(
    sample_patients_df,
    total_data,
    output_file='patients_with_formats.json'
)

Processing 32 patients...
Patient 37152/32: 16380214_20860234
Patient 9957/32: 11719852_29694624
Patient 107/32: 10017882_20281726
Patient 46879/32: 18021792_23272302
Patient 26087/32: 14486442_24893935
Patient 48765/32: 18360819_27033030
Patient 28461/32: 14898288_21598790
Patient 16343/32: 12789235_26659628
Patient 3716/32: 10652693_28272409
Patient 35631/32: 16124481_28174564
Patient 21459/32: 13697787_22246602
Patient 25641/32: 14413352_20235742
Patient 55313/32: 19450100_25314766
Patient 9559/32: 11653128_24553816
Patient 38181/32: 16536632_27119150
Patient 53921/32: 19231238_21424071
Patient 47651/32: 18151496_21104362
Patient 4940/32: 10857992_22199407
Patient 17534/32: 13004228_29566269
Patient 16475/32: 12815857_25911030
Patient 33639/32: 15794497_28987036
Patient 56441/32: 19654579_23780957
Patient 11424/32: 11961951_21282341
Patient 28261/32: 14872672_29294271
Patient 37273/32: 16393879_28139528
Patient 42635/32: 17306425_21245006
Patient 31053/32: 15363935_24356132
Patient 

### Getting SHAP+background values

In [11]:
import pandas as pd
deltas = pd.read_csv("total_deltas.csv")
features_to_analyze = deltas['feature'].unique()

shap_bck = pd.read_csv("shap_matrix_background.csv")
shap_abs_bck = pd.read_csv("shap_matrix_abs_background.csv")
shap_long_bck = shap_bck.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean')
shap_abs_long_bck = shap_abs_bck.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean_abs')
shap_combined_bck = shap_long_bck.merge(shap_abs_long_bck, on=['hadm_id', 'feature'], how='inner')

shap_combined_bck = shap_combined_bck[shap_combined_bck['feature'].isin(features_to_analyze)]

In [12]:
import json
with open('patients_with_formats.json', 'r', encoding='utf-8') as f:
    ehrs = json.load(f)

ehr_lookup = {
    (x["subject_id"], x["hadm_id"]): x["json_context"]
    for x in ehrs['patients']
}

clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 145}, #sodium
    'lab_50971_daily': {'low': 3.5, 'high': 5}, #potassium
    'lab_50902_daily': {'low': 95, 'high': 105}, #chloride
    'lab_50882_daily': {'low': 22, 'high': 32}, #bicarbonate
    'lab_50912_daily': {'low': 0.8, 'high': 1.3}, #creatinine
    'lab_51006_daily': {'low': 8, 'high': 21}, #BUN
    'lab_50931_daily': {'low': 65, 'high': 110}, #glucose
    'lab_50893_daily': {'low': 8.5, 'high': 10.2}, #calcium
    'lab_50868_daily': {'low': 10, 'high': 18}, #anion gap
    'lab_51222_daily': {
        'male': {'low': 13, 'high': 17},
        'female': {'low': 12, 'high': 15}
    }, #hemoglobin
    'lab_51301_daily': {'low': 4, 'high': 10}, #WBC
    'lab_51265_daily': {'low': 150, 'high': 400}, #Platelet Count
    'lab_51221_daily': {
        'male': {'low': 40, 'high': 52},
        'female': {'low': 36, 'high': 47}
    }, #Hematocrit
    'lab_51250_daily': {'low': 80, 'high': 100}, #MCV
    'lab_51277_daily': {'low': 11.5, 'high': 14.5}, #RDW
    'lab_50960_daily': {'low': 1.5, 'high': 2}, #magnesium
    'lab_50970_daily': {'low': 2.7, 'high': 4.5}, #phosphate
    'lab_51248_daily': {'low': 26, 'high': 32}, #MCH
    'lab_51249_daily': {'low': 30, 'high': 35}, #MCHC
    'lab_51279_daily': {
        'male': {'low': 4.5, 'high': 5.9},
        'female': {'low': 4, 'high': 5.2}
    } #RBC
}

In [13]:
def correct_shap_signs(top_features, ehr_info):
    corrected_features = {}
    
    is_male = ehr_info.get("gender_male", 0)
    gender_key = "male" if (is_male == 1 or is_male is True) else "female"
    
    patient_labs = ehr_info.get('laboratory_values', {})
    clinical_indicators = ehr_info.get('clinical_indicators', {})

    for feature, shap_val in top_features.items():
        corrected_val = shap_val
        
        always_increase_risk_features = [
            'age', 
            'Obesity (END009)', 
            'Heart failure (CIR019)', 
            'Hyperlipidemia, unspecified (E785)', 
            'Disorders of lipid metabolism (END010)'
            'Diabetes mellitus with complication (END003)'
        ]
        if any(x.lower() in feature.lower() for x in always_increase_risk_features):
            if shap_val < 0:
                corrected_val = abs(shap_val)

        elif 'medications' in feature.lower() or 'procedures' in feature.lower():
            actual_val = clinical_indicators.get(feature, 0)
                
            if actual_val == 0 and shap_val > 0:
                corrected_val = -abs(shap_val)

        elif feature in ['comorbidity_score', 'num_chronic']:
            actual_indicator_str = clinical_indicators.get(feature)
            if actual_indicator_str is not None:
                actual_ind_val = float(actual_indicator_str)
                thresholds = {
                    'comorbidity_score': 3,
                    'num_chronic': 2
                }
                max_safe_value = thresholds.get(feature, 0)
                if 0 <= actual_ind_val <= max_safe_value:
                    if shap_val > 0:
                        corrected_val = -abs(shap_val)

        else:
            import re
            lab_id_match = re.search(r'\((\d{5})\)', feature)
            
            if lab_id_match:
                lab_id = lab_id_match.group(1)
                range_key = f"lab_{lab_id}_daily"
                
                if range_key in clinical_ranges:
                    actual_lab_str = patient_labs.get(feature)
                    
                    if actual_lab_str is not None:
                        actual_lab_val = float(actual_lab_str)
                        norm_data = clinical_ranges[range_key]
                        
                        if 'male' in norm_data or 'female' in norm_data:
                            low_limit = norm_data[gender_key]['low']
                            high_limit = norm_data[gender_key]['high']
                        else:
                            low_limit = norm_data['low']
                            high_limit = norm_data['high']
                        
                        if low_limit <= actual_lab_val <= high_limit:
                            if shap_val > 0:
                                corrected_val = -abs(shap_val)

        corrected_features[feature] = corrected_val

    return corrected_features

In [14]:
shap_combined_bck['feature'] = shap_combined_bck['feature'].apply(map_feature_name)
shap_fin = []
for idx, row in sample_patients_df.iterrows():
    subject_id = row['subject_id']
    hadm_id = row['hadm_id']
    shap_cur = shap_combined_bck[shap_combined_bck['hadm_id'] == hadm_id].sort_values(by='shap_mean_abs', ascending=False)
    shap_dict = shap_cur.set_index('feature')['shap_mean'].to_dict()

    ehr_info = ehr_lookup.get((subject_id, hadm_id))
    diagnoses_icd = ehr_info['diagnoses'].get('icd', [])
    diagnoses_ccsr = ehr_info['diagnoses'].get('ccsr', [])
    lab_vals = list(ehr_info.get('laboratory_values', {}).keys())
    other_features = [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]

    allowed_features = set(diagnoses_icd + diagnoses_ccsr + lab_vals + other_features)
    shap_top_bck = {
        feature: value
        for feature, value in shap_dict.items()
        if feature in allowed_features
    }
    top_features = dict(list(shap_top_bck.items())[:10])
    top_features_fin = correct_shap_signs(top_features, ehr_info)
    patient_entry = {
        'subject_id': subject_id,
        'hadm_id': hadm_id,
        'shap_bck_values': top_features_fin,
    }
    shap_fin.append(patient_entry)
with open('shap_bck_all_patients_dev.json', 'w', encoding='utf-8') as f:
    json.dump(shap_fin, f, ensure_ascii=False, indent=2)